In [10]:
import sys, gc, os
sys.path.append("..")
import pandas as pd
from src.data_loader import download_and_load_brfss

shortlist = ["_MICHD", "_AGE_G", "SEXVAR", "_RACE", "EDUCA", "INCOME3",
             "GENHLTH", "_BMI5", "DIABETE4", "_RFHYPE6", "TOLDHI3",
             "_SMOKER3", "_TOTINDA", "SLEPTIM1", "_RFDRHV8"]

try:
    df_full = df
    assert df_full.shape[1] > 100
    print("Reusing the dataframe already in memory")
except (NameError, AssertionError):
    print("Downloading from the CDC (a minute or two)...")
    df_full = download_and_load_brfss()

print(df_full.shape)    # expect (457670, 301)
df.head()

(457670, 301)


,_MICHD,_AGE_G,SEXVAR,_RACE,EDUCA,INCOME3,GENHLTH,_BMI5,DIABETE4,_SMOKER3,_TOTINDA
0,2.0,6.0,2.0,1.0,4.0,99.0,3.0,2249.0,3.0,4.0,1.0
1,1.0,6.0,1.0,1.0,6.0,11.0,1.0,2583.0,3.0,3.0,1.0
2,2.0,5.0,1.0,1.0,5.0,99.0,2.0,2253.0,3.0,1.0,1.0
3,2.0,6.0,1.0,1.0,6.0,6.0,1.0,2509.0,3.0,4.0,1.0
4,2.0,4.0,1.0,1.0,5.0,3.0,3.0,1977.0,3.0,4.0,2.0


In [11]:
missing = [c for c in shortlist if c not in df_full.columns]
print("Missing columns:", missing)     # hopefully []

df = df_full[[c for c in shortlist if c in df_full.columns]].copy()
del df_full
gc.collect()

print(df.shape)                         # expect (457670, 15) if nothing is missing
df.head()

Missing columns: ['_RFHYPE6', 'TOLDHI3', 'SLEPTIM1', '_RFDRHV8']
(457670, 11)


,_MICHD,_AGE_G,SEXVAR,_RACE,EDUCA,INCOME3,GENHLTH,_BMI5,DIABETE4,_SMOKER3,_TOTINDA
0,2.0,6.0,2.0,1.0,4.0,99.0,3.0,2249.0,3.0,4.0,1.0
1,1.0,6.0,1.0,1.0,6.0,11.0,1.0,2583.0,3.0,3.0,1.0
2,2.0,5.0,1.0,1.0,5.0,99.0,2.0,2253.0,3.0,1.0,1.0
3,2.0,6.0,1.0,1.0,6.0,6.0,1.0,2509.0,3.0,4.0,1.0
4,2.0,4.0,1.0,1.0,5.0,3.0,3.0,1977.0,3.0,4.0,2.0


In [14]:
import os, gc

shortlist = [
    "_MICHD",
    # core
    "_AGE_G", "SEXVAR", "_RACE", "EDUCA", "INCOME3", "GENHLTH", "_BMI5",
    "DIABETE4", "_SMOKER3", "_TOTINDA", "_RFDRHV9",
    # chronic conditions
    "CHCKDNY2", "CHCCOPD3", "ASTHMA3", "ADDEPEV3", "CHCSCNC1", "CHCOCNC1", "HAVARTH4",
    # health and function
    "PHYSHLTH", "MENTHLTH", "DIFFWALK", "DIFFDRES", "DECIDE",
    # lifestyle
    "DRNKANY6", "_DRNKWK3",
    # social and access
    "MARITAL", "EMPLOY1", "MEDCOST1", "CHECKUP1",
]

missing = [c for c in shortlist if c not in df_full.columns]
print("Missing:", missing)           # must be []
assert not missing

df = df_full[shortlist].copy()
del df_full
gc.collect()

print(df.shape)                       # expect (457670, 30)

os.makedirs("../data/raw", exist_ok=True)
df.to_parquet("../data/raw/brfss2024_subset.parquet", index=False)
print(f"Saved: {os.path.getsize('../data/raw/brfss2024_subset.parquet')/1e6:.1f} MB")

Missing: []
(457670, 30)
Saved: 5.0 MB


In [15]:
for f in ["../data/raw/full_2024.parquet", "../data/raw/codebook24.zip", "data/raw/codebook24.zip"]:
    if os.path.exists(f):
        os.remove(f)
        print("Deleted", f)

In [16]:
print(df["_MICHD"].value_counts(dropna=False))

_MICHD
2.0    410126
1.0     42338
NaN      5206
Name: count, dtype: int64
